In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
lineups = pd.read_csv("using_dataset/lineups_five_seasons.csv")
lineups = lineups[lineups["POSS"] >= 100]
lineups.head()

,GROUP_SET,GROUP_ID,GROUP_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,W_PCT,MIN,...,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SUM_TIME_PLAYED,SEASON
0,Lineups,-201145-203115-203932-203999-1628420-,J. Green - W. Barton - A. Gordon - N. Jokic - ...,1610612743,DEN,41,26,15,0.634,761.0,...,6705,7252,6370,10452,5593,5677,11236,6814,2283635,2021-22
1,Lineups,-101108-203109-1626164-1628969-1629028-,C. Paul - J. Crowder - D. Booker - M. Bridges ...,1610612756,PHX,38,31,7,0.816,754.0,...,7902,8158,9261,9044,6230,6504,11310,7014,2262330,2021-22
2,Lineups,-201144-202711-203497-1626220-1628378-,M. Conley - B. Bogdanovic - R. Gobert - R. O'N...,1610612762,UTA,42,29,13,0.690,666.0,...,6620,8318,6570,10529,5726,5990,11114,7177,1998085,2021-22
3,Lineups,-201143-203935-1627759-1628369-1629057-,A. Horford - M. Smart - J. Brown - J. Tatum - ...,1610612738,BOS,34,27,7,0.794,443.0,...,6625,7212,5935,10439,5608,6078,11924,5394,1328350,2021-22
4,Lineups,-201976-1626156-1626157-1629020-1630162-,P. Beverley - D. Russell - K. Towns - J. Vande...,1610612750,MIN,36,22,14,0.611,439.0,...,4591,8705,6434,10517,6238,6743,11678,7262,1316725,2021-22


In [3]:
players = pd.read_csv("using_dataset/full_data.csv")
players.head()

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,SYNERGY_OFFENSIVE_ISOLATION_PCT,SYNERGY_OFFENSIVE_MISC_PCT,SYNERGY_OFFENSIVE_OFFREBOUND_PCT,SYNERGY_OFFENSIVE_OFFSCREEN_PCT,SYNERGY_OFFENSIVE_POSTUP_PCT,SYNERGY_OFFENSIVE_PRBALLHANDLER_PCT,SYNERGY_OFFENSIVE_PRROLLMAN_PCT,SYNERGY_OFFENSIVE_SPOTUP_PCT,SYNERGY_OFFENSIVE_TRANSITION_PCT,CLUSTER
0,203932,Aaron Gordon,Aaron,1610612743,DEN,26.0,75,46,29,0.613,...,0.073239,0.058216,0.075117,0.015023,0.147418,0.047887,0.047887,0.175587,0.190610,3
1,1628988,Aaron Holiday,Aaron,1610612756,PHX,25.0,63,34,29,0.540,...,0.050633,0.065823,0.000000,0.000000,0.000000,0.387342,0.000000,0.301266,0.154430,1
2,1630174,Aaron Nesmith,Aaron,1610612738,BOS,22.0,52,32,20,0.615,...,0.000000,0.058824,0.000000,0.101604,0.000000,0.000000,0.000000,0.641711,0.197861,3
3,1630598,Aaron Wiggins,Aaron,1610612760,OKC,23.0,50,13,37,0.260,...,0.024272,0.036408,0.070388,0.000000,0.000000,0.111650,0.000000,0.402913,0.233010,3
4,1627846,Abdel Nader,Abdel,1610612756,PHX,28.0,14,11,3,0.786,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.576923,0.423077,4


In [4]:
stat_cols = [
    "USG_PCT",
    "AST_PCT",
    "REB_PCT",
    "TOV_PCT",
    "TS_PCT",
    "EFG_PCT",
    "FG3A",
    "FG3_PCT",
    "STL",
    "BLK"
]

off_playtype_cols = [
    col for col in players.columns
    if col.startswith("SYNERGY_OFF") and col.endswith("_PCT")
]

def_playtype_cols = [
    col for col in players.columns
    if col.startswith("SYNERGY_DEF") and col.endswith("_PCT")
]

print("Offensive play types:", len(off_playtype_cols))
print("Defensive play types:", len(def_playtype_cols))
print("Traditional statistics:", len(stat_cols))

Offensive play types: 11
Defensive play types: 7
Traditional statistics: 10


In [5]:
duplicates = players[
    players.duplicated(
        subset=["PLAYER_ID", "SEASON"],
        keep=False
    )
]

print("Duplicate player-season rows:", len(duplicates))

Duplicate player-season rows: 0


In [6]:
player_id_cols = [
    "PLAYER_1_ID",
    "PLAYER_2_ID",
    "PLAYER_3_ID",
    "PLAYER_4_ID",
    "PLAYER_5_ID"
]

# Only needed if your lineup dataset uses GROUP_ID
def parse_group_id(group_id):
    ids = str(group_id).strip("-").split("-")
    int_ids = [int(id) for id in ids]

    if len(int_ids) != 5:
        return [None] * 5

    return int_ids

parsed_ids = lineups["GROUP_ID"].apply(parse_group_id)

for i in range(5):
    lineups[player_id_cols[i]] = parsed_ids.apply(
        lambda x: x[i]
    )

parsed_ids

0           [201145, 203115, 203932, 203999, 1628420]
1         [101108, 203109, 1626164, 1628969, 1629028]
2          [201144, 202711, 203497, 1626220, 1628378]
3         [201143, 203935, 1627759, 1628369, 1629057]
4        [201976, 1626156, 1626157, 1629020, 1630162]
                            ...                      
8273    [1629628, 1630193, 1630567, 1630572, 1641711]
8277     [203083, 1629656, 1630178, 1631230, 1642845]
8283    [1626157, 1628384, 1628404, 1628973, 1629013]
8286    [1629655, 1630230, 1630314, 1631108, 1642843]
8298    [1629638, 1630552, 1630700, 1631243, 1642258]
Name: GROUP_ID, Length: 1492, dtype: object

In [7]:
lineups.head()

,GROUP_SET,GROUP_ID,GROUP_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,W_PCT,MIN,...,TS_PCT_RANK,PACE_RANK,PIE_RANK,SUM_TIME_PLAYED,SEASON,PLAYER_1_ID,PLAYER_2_ID,PLAYER_3_ID,PLAYER_4_ID,PLAYER_5_ID
0,Lineups,-201145-203115-203932-203999-1628420-,J. Green - W. Barton - A. Gordon - N. Jokic - ...,1610612743,DEN,41,26,15,0.634,761.0,...,5677,11236,6814,2283635,2021-22,201145,203115,203932,203999,1628420
1,Lineups,-101108-203109-1626164-1628969-1629028-,C. Paul - J. Crowder - D. Booker - M. Bridges ...,1610612756,PHX,38,31,7,0.816,754.0,...,6504,11310,7014,2262330,2021-22,101108,203109,1626164,1628969,1629028
2,Lineups,-201144-202711-203497-1626220-1628378-,M. Conley - B. Bogdanovic - R. Gobert - R. O'N...,1610612762,UTA,42,29,13,0.690,666.0,...,5990,11114,7177,1998085,2021-22,201144,202711,203497,1626220,1628378
3,Lineups,-201143-203935-1627759-1628369-1629057-,A. Horford - M. Smart - J. Brown - J. Tatum - ...,1610612738,BOS,34,27,7,0.794,443.0,...,6078,11924,5394,1328350,2021-22,201143,203935,1627759,1628369,1629057
4,Lineups,-201976-1626156-1626157-1629020-1630162-,P. Beverley - D. Russell - K. Towns - J. Vande...,1610612750,MIN,36,22,14,0.611,439.0,...,6743,11678,7262,1316725,2021-22,201976,1626156,1626157,1629020,1630162


In [8]:
def create_lineup_dataset(lineups, players, player_id_cols, stat_cols, off_playtype_cols, def_playtype_cols, cluster_col="CLUSTER"):
    # Combine all individual player features
    feature_cols = (
        stat_cols
        + off_playtype_cols
        + def_playtype_cols
    )

    # Only keep columns that actually exist
    feature_cols = [
        col for col in feature_cols
        if col in players.columns
    ]

    # Find all possible archetypes
    clusters = sorted(players[cluster_col].dropna().unique())

    # Create a lookup dictionary:
    # (PLAYER_ID, SEASON) -> player statistics
    player_lookup = players.set_index(
        ["PLAYER_ID", "SEASON"]
    )

    lineup_rows = []

    for _, lineup in lineups.iterrows():

        season = lineup["SEASON"]

        # Get the five player IDs
        ids = [
            lineup[col]
            for col in player_id_cols
        ]

        # Retrieve all five player profiles
        profiles = []

        for player_id in ids:
            key = (player_id, season)

            if key not in player_lookup.index:
                profiles = []
                break

            profiles.append(player_lookup.loc[key])

        # Require all five players to have matching profiles
        if len(profiles) != 5:
            continue

        player_profiles = pd.DataFrame(profiles)

        # Start constructing this lineup's feature row
        row = {
            "SEASON": season,
            "GROUP_ID": lineup.get("GROUP_ID", None)
        }

        # Preserve lineup metadata
        metadata_cols = [
            "MIN",
            "POSS",
            "GP",
            "OFF_RATING",
            "DEF_RATING",
            "NET_RATING"
        ]
        
        for col in metadata_cols:
            if col in lineups.columns:
                row[col] = lineup[col]

        # ---------------------------------
        # 1. Cluster count / multi-hot features
        # ---------------------------------

        cluster_counts = (
            player_profiles[cluster_col]
            .value_counts()
        )

        for cluster in clusters:
            row[f"CLUSTER_{cluster}"] = (
                cluster_counts.get(cluster, 0)
            )

        # ---------------------------------
        # 2. Average play-type percentages
        # ---------------------------------

        for col in off_playtype_cols + def_playtype_cols:

            if col in player_profiles.columns:
                row[f"AVG_{col}"] = (
                    player_profiles[col].mean()
                )

                # Optional: variation among the five players
                row[f"STD_{col}"] = (
                    player_profiles[col].std()
                )

        # ---------------------------------
        # 3. Traditional / advanced statistics
        # ---------------------------------

        for col in stat_cols:

            if col in player_profiles.columns:
                row[f"AVG_{col}"] = (
                    player_profiles[col].mean()
                )

                # Optional variation
                row[f"STD_{col}"] = (
                    player_profiles[col].std()
                )

        lineup_rows.append(row)

    return pd.DataFrame(lineup_rows)

In [9]:
cleaned_data = create_lineup_dataset(
    lineups,
    players,
    player_id_cols,
    stat_cols,
    off_playtype_cols,
    def_playtype_cols,
    cluster_col="CLUSTER"
)

cleaned_data.head()

,SEASON,GROUP_ID,MIN,POSS,GP,OFF_RATING,DEF_RATING,NET_RATING,CLUSTER_0,CLUSTER_1,...,AVG_EFG_PCT,STD_EFG_PCT,AVG_FG3A,STD_FG3A,AVG_FG3_PCT,STD_FG3_PCT,AVG_STL,STD_STL,AVG_BLK,STD_BLK
0,2021-22,-201145-203115-203932-203999-1628420-,761.0,1589,41,121.3,111.8,9.5,1,2,...,0.5712,0.033833,3.98,1.409610,0.3494,0.031093,0.80,0.418330,0.50,0.264575
1,2021-22,-101108-203109-1626164-1628969-1629028-,754.0,1577,38,116.4,109.0,7.4,1,2,...,0.5644,0.053003,3.92,2.523292,0.3570,0.025603,1.26,0.439318,0.44,0.151658
2,2021-22,-201144-202711-203497-1626220-1628378-,666.0,1395,42,117.1,110.5,6.7,1,2,...,0.5862,0.074254,5.30,3.587478,0.3078,0.173115,1.02,0.414729,0.60,0.851469
3,2021-22,-201143-203935-1627759-1628369-1629057-,443.0,911,34,118.8,94.2,24.6,2,3,...,0.5700,0.094433,4.90,3.292416,0.2756,0.154478,1.08,0.376829,0.94,0.814248
4,2021-22,-201976-1626156-1626157-1629020-1630162-,439.0,906,36,120.2,107.4,12.8,1,3,...,0.5422,0.045307,5.14,3.322349,0.3186,0.102124,1.20,0.212132,0.70,0.308221


In [10]:
cleaned_data.columns

Index(['SEASON', 'GROUP_ID', 'MIN', 'POSS', 'GP', 'OFF_RATING', 'DEF_RATING',
       'NET_RATING', 'CLUSTER_0', 'CLUSTER_1', 'CLUSTER_2', 'CLUSTER_3',
       'CLUSTER_4', 'CLUSTER_5', 'AVG_SYNERGY_OFFENSIVE_CUT_PCT',
       'STD_SYNERGY_OFFENSIVE_CUT_PCT', 'AVG_SYNERGY_OFFENSIVE_HANDOFF_PCT',
       'STD_SYNERGY_OFFENSIVE_HANDOFF_PCT',
       'AVG_SYNERGY_OFFENSIVE_ISOLATION_PCT',
       'STD_SYNERGY_OFFENSIVE_ISOLATION_PCT', 'AVG_SYNERGY_OFFENSIVE_MISC_PCT',
       'STD_SYNERGY_OFFENSIVE_MISC_PCT',
       'AVG_SYNERGY_OFFENSIVE_OFFREBOUND_PCT',
       'STD_SYNERGY_OFFENSIVE_OFFREBOUND_PCT',
       'AVG_SYNERGY_OFFENSIVE_OFFSCREEN_PCT',
       'STD_SYNERGY_OFFENSIVE_OFFSCREEN_PCT',
       'AVG_SYNERGY_OFFENSIVE_POSTUP_PCT', 'STD_SYNERGY_OFFENSIVE_POSTUP_PCT',
       'AVG_SYNERGY_OFFENSIVE_PRBALLHANDLER_PCT',
       'STD_SYNERGY_OFFENSIVE_PRBALLHANDLER_PCT',
       'AVG_SYNERGY_OFFENSIVE_PRROLLMAN_PCT',
       'STD_SYNERGY_OFFENSIVE_PRROLLMAN_PCT',
       'AVG_SYNERGY_OFFENSIVE_SPOTUP_P

In [11]:
cleaned_data.shape

(1476, 68)